#### Cell 1 — load loop_augmented_truncated.xlsx, project to the 5-column schema

In [1]:
import pandas as pd
from pathlib import Path

loop = pd.read_excel("../loop_augmented_truncated.xlsx")
print(f"Loaded loop_augmented_truncated.xlsx: {loop.shape}")

training_df = loop[["Trace ID", "Trace Content", "Verified Label", "Source", "Parent Trace ID"]].copy()
training_df = training_df.rename(columns={"Verified Label": "Label"})

print(f"\nProjected shape: {training_df.shape}")
print(training_df.columns.tolist())
print()
print(training_df["Label"].value_counts())
print()
print(training_df["Source"].value_counts())
print()
print("Nulls per column:")
print(training_df.isna().sum())
print()
print("Unique Trace IDs:", training_df["Trace ID"].is_unique)

Loaded loop_augmented_truncated.xlsx: (146, 10)

Projected shape: (146, 5)
['Trace ID', 'Trace Content', 'Label', 'Source', 'Parent Trace ID']

Label
LOOP    146
Name: count, dtype: int64

Source
SYNTHETIC              56
SYNTHETIC_TRUNCATED    56
REAL_TRUNCATED         19
REAL                   15
Name: count, dtype: int64

Nulls per column:
Trace ID           0
Trace Content      0
Label              0
Source             0
Parent Trace ID    0
dtype: int64

Unique Trace IDs: True


#### Cell 2 — add unsafe_execution_final.xlsx

In [2]:
ue = pd.read_excel("../unsafe_execution_final.xlsx")
print(f"Loaded unsafe_execution_final.xlsx: {ue.shape}")

ue_proj = ue[["Trace ID", "Trace Content", "Verified Label", "Source", "Parent Trace ID"]].copy()
ue_proj = ue_proj.rename(columns={"Verified Label": "Label"})

# check for ID collisions before merging
overlap = set(training_df["Trace ID"]) & set(ue_proj["Trace ID"])
print(f"\nID collisions with existing training_df: {len(overlap)}")

training_df = pd.concat([training_df, ue_proj], ignore_index=True)

print(f"\nCombined shape: {training_df.shape}")
print(training_df["Label"].value_counts())
print()
print("Unique Trace IDs:", training_df["Trace ID"].is_unique)
print("Duplicate Trace Content:", training_df["Trace Content"].duplicated().sum())
print("Nulls per column:")
print(training_df.isna().sum())

Loaded unsafe_execution_final.xlsx: (208, 11)

ID collisions with existing training_df: 0

Combined shape: (354, 5)
Label
UNSAFE_EXECUTION    208
LOOP                146
Name: count, dtype: int64

Unique Trace IDs: True
Duplicate Trace Content: 0
Nulls per column:
Trace ID           0
Trace Content      0
Label              0
Source             0
Parent Trace ID    0
dtype: int64


#### Cell 3 — add SUCCESS/HALLUCINATION, backfill Source/Parent Trace ID, save

In [3]:
master = pd.read_excel("../trace_annotation_log.xlsx")
print(f"Loaded trace_annotation_log.xlsx: {master.shape}")
print(master["Verified Label"].value_counts())

# critical: only SUCCESS/HALLUCINATION -- LOOP/UNSAFE_EXECUTION here are pre-augmentation
# originals already superseded by the two files above (see prior duplication finding)
sh = master[master["Verified Label"].isin(["SUCCESS", "HALLUCINATION"])].copy()
print(f"\nSUCCESS/HALLUCINATION rows: {len(sh)}")

sh_proj = sh[["Trace ID", "Trace Content", "Verified Label"]].copy()
sh_proj = sh_proj.rename(columns={"Verified Label": "Label"})
sh_proj["Source"] = "REAL"
sh_proj["Parent Trace ID"] = sh_proj["Trace ID"]

overlap = set(training_df["Trace ID"]) & set(sh_proj["Trace ID"])
print(f"ID collisions with existing training_df: {len(overlap)}")

training_df = pd.concat([training_df, sh_proj], ignore_index=True)

print(f"\nFinal shape: {training_df.shape}")
print(training_df["Label"].value_counts())
print()
print("Unique Trace IDs:", training_df["Trace ID"].is_unique)
print("Duplicate Trace Content:", training_df["Trace Content"].duplicated().sum())
print("Nulls per column:")
print(training_df.isna().sum())
print()
print("e27ea2e2 present (should be False):", "e27ea2e2" in training_df["Trace ID"].values)

training_df.to_excel("../training_dataset.xlsx", index=False)
print("\nSaved to ../training_dataset.xlsx")

Loaded trace_annotation_log.xlsx: (267, 8)
Verified Label
SUCCESS             129
HALLUCINATION       109
LOOP                 16
UNSAFE_EXECUTION     13
Name: count, dtype: int64

SUCCESS/HALLUCINATION rows: 238
ID collisions with existing training_df: 0

Final shape: (592, 5)
Label
UNSAFE_EXECUTION    208
LOOP                146
SUCCESS             129
HALLUCINATION       109
Name: count, dtype: int64

Unique Trace IDs: True
Duplicate Trace Content: 0
Nulls per column:
Trace ID           0
Trace Content      0
Label              0
Source             0
Parent Trace ID    0
dtype: int64

e27ea2e2 present (should be False): False

Saved to ../training_dataset.xlsx
